In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

import os
import cv2

import tensorflow as tf
import pickle
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.models import Sequential


In [ ]:
data_path = "/kaggle/input/datasets/sumn2u/garbage-classification-v2/original"


biological_path= os.path.join(data_path, "biological")
cardboard_path= os.path.join(data_path, "cardboard")
clothes_path= os.path.join(data_path, "clothes")
glass_path= os.path.join(data_path, "glass")
metal_path= os.path.join(data_path, "metal")
paper_path= os.path.join(data_path, "paper")
plastic_path= os.path.join(data_path, "plastic")
shoes_path= os.path.join(data_path, "shoes")



In [ ]:
data = []
labels = []
categories = [
    "cardboard","clothes",
    "glass","metal","paper","plastic","shoes"
]

for label, category in enumerate(categories):
    path = os.path.join(data_path, category)
    count = 0  

    for file in os.listdir(path):
        if count >= 756:  
            break
        try:
            img_path = os.path.join(path, file)
            img = cv2.imread(img_path)

            if img is None:
                continue

            img = cv2.resize(img, (128, 128))
            data.append(img)
            labels.append(label)
            count += 1
        except:
            pass

In [ ]:
from collections import Counter
print(Counter(labels))

In [ ]:
data= np.array(data)/255.0
labels= np.array(labels)

In [ ]:
print(data.shape)


In [ ]:
print(labels.shape)

In [ ]:
X_train , X_test , y_train , y_test = train_test_split(
    data , labels , test_size=0.2 , random_state=42
)
print(X_train.shape)
print(y_train.shape)
print(X_test.shape)
print(y_test.shape)

In [ ]:
model = Sequential()

model.add(Conv2D(32,(3,3),activation="relu",input_shape=(128,128,3)))
model.add(MaxPooling2D(2,2))

model.add(Conv2D(64,(3,3),activation="relu"))
model.add(MaxPooling2D(2,2))

model.add(Conv2D(128,(3,3),activation="relu"))
model.add(MaxPooling2D(2,2))

model.add(Flatten())
model.add(Dense(128,activation="relu"))
model.add(Dropout(0.5))

model.add(Dense(10,activation="softmax"))

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

h = model.fit(
    X_train,
    y_train,
    epochs=10,
    batch_size=32,
    validation_data=(X_test,y_test)
)

In [ ]:
plt.plot(h.history['accuracy'])
plt.plot(h.history['val_accuracy'])

plt.title("Model Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")

plt.legend(["Train", "Validation"])
plt.show()

In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model

base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(128,128,3)
)

for layer in base_model.layers:
    layer.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)

x = Dense(128, activation="relu")(x)
x = Dropout(0.5)(x)

output = Dense(7, activation="softmax")(x)

model = Model(inputs=base_model.input, outputs=output)

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history = model.fit(
    X_train,
    y_train,
    epochs=10,
    batch_size=32,
    validation_data=(X_test,y_test)
)

In [ ]:
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])

plt.title("Model Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")

plt.legend(["Train", "Validation"])
plt.show()

In [ ]:
model.save("Garbage_Classifier.h5")